<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/Anthropic_API_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q anthropic openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 15.8 MB/s eta 0:00:00


In [16]:
from anthropic import Anthropic
from google.colab import userdata
from openai import OpenAI
from pathlib import Path
from pydantic import BaseModel

#TODO TYPING
def extract_summary(message):
  result = [block.text for block in message.content if block.type == 'text']
  return "\n\n\n".join(result)

# TODO: improve typing: Use list comprehension
def extract_keywords(response):
  result = []
  for item in response.output:
      if item.type == 'message':
          for block in item.content:
              if block.type == 'output_text':
                  result.append(block.text)

  return "\n\n\n".join(result)

anthropic_api_key = userdata.get("ANTHROPIC_API_KEY")
anthropic_client = Anthropic(api_key=anthropic_api_key)

openai_api_key= userdata.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key = openai_api_key)

openrouter_api_key= userdata.get("OPENROUTER_API_KEY")
openrouter_client = OpenAI(api_key = openrouter_api_key, base_url="https://openrouter.ai/api/v1")

In [29]:
SUMMARIZE_SYSTEM_PROMPT = """You are an expert at summarizing legal and technical documents.

Produce a faithful summary of the document provided.

Length: aim for ~200 words. This is a target, not a hard limit—if covering all important facts requires more words, exceed it. Never omit an important fact to hit the word count.

For legal and technical documents, "important facts" typically include: parties and their roles, obligations and rights, conditions and contingencies, deadlines and effective dates, monetary amounts, liabilities, termination or breach terms, defined terms, and any explicit exceptions or limitations.

Fidelity rules:
- Every statement must be directly supported by the text. Do not infer, extrapolate, or add outside knowledge.
- Do not editorialize or offer opinions on whether terms are fair, standard, or advisable.
- Preserve the meaning of legal/technical terms of art; do not paraphrase them into something looser or incorrect.
- If the document is ambiguous or silent on a point, do not resolve it—either omit it or note the ambiguity briefly.

Write in clear, plain prose. Use short paragraphs or bullet points where that aids readability."""

CATEGORIZE_SYSTEM_PROMPT = """You are an expert in categorizing legal and technical documents.

## Task
You will receive a short summary of a document. Based solely on the information in that summary, assign the document to the most appropriate category.

## Instructions
- Read the summary carefully and identify the document's primary purpose, subject matter, and type.
- Choose the single category that best fits the document.
- If the summary is ambiguous, sparse, or could fit multiple categories, reflect this uncertainty in a lower confidence score rather than guessing arbitrarily.
- Base your decision only on the provided summary. Do not invent details that are not present.

## Confidence Score
Output a confidence score as a decimal in the range [0, 1], where:
- 1.0 = complete certainty in the assigned category
- 0.5 = the category is plausible but the summary is ambiguous or supports alternatives
- 0.0 = no meaningful basis for the categorization

## Output Format
Respond with a JSON object and nothing else:
{
  "category": "<chosen category>",
  "confidence": <float between 0 and 1>
}
"""

class DocumentCategorization(BaseModel):
  category: str
  confidence: float


In [32]:
def analyze_document(path_to_file: Path):
    create_file_response = None

    try:
        with path_to_file.open("rb") as file:
            create_file_response = anthropic_client.beta.files.upload(
                file=(path_to_file.name, file, "application/pdf"),
                betas=["files-api-2025-04-14"],
            )

        summarize_response = anthropic_client.beta.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=4096,
            system=SUMMARIZE_SYSTEM_PROMPT,
            betas=["files-api-2025-04-14"],
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "document",
                            "source": {
                                "type": "file",
                                "file_id": create_file_response.id,
                            },
                        },
                        {
                            "type": "text",
                            "text": "Summarize the referenced document.",
                        },
                    ],
                }
            ],
        )

        summary = extract_summary(summarize_response)
        print("-" * 50)
        print("Summary:")
        print(summary)

        extract_keywords_response = openai_client.responses.create(
            model="gpt-5-mini",
            input=[
                {
                    "role": "system",
                    "content": (
                        "You are an expert in keyword extraction. "
                        "Given a document summary, return a list of maximum 10 "
                        "important keywords. Focus on terms, named entities, "
                        "and domain-specific vocabulary."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Extract keywords from this summary:\n\n{summary}",
                },
            ],
            reasoning={"effort": "high"},
        )

        keywords = extract_keywords(extract_keywords_response)
        print("-" * 50)
        print("Keywords:")
        print(keywords)

        categorize_response = openrouter_client.responses.parse(
            model="deepseek/deepseek-v4-flash",
            input=[
                {"role": "system", "content": CATEGORIZE_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"Categorize this document based on its summary:\n\n{summary}",
                },
            ],
            text_format=DocumentCategorization,
        )

        categorization = categorize_response.output_parsed
        print("-" * 50)
        print("Categorization:")
        print(categorization)

        return summary, keywords, categorization

    finally:
        if create_file_response is not None:
            anthropic_client.beta.files.delete(
                file_id=create_file_response.id,
                betas=["files-api-2025-04-14"],
            )

In [33]:
analyze_document(Path("/content/sample_data/test_doc_02.pdf"))

--------------------------------------------------
Summary:
# Summary: Data Migration Assistant – AI-Assisted Development Project

## Project Overview
Data Migration Assistant is a web-based application designed to simplify migration of spreadsheet data into PostgreSQL databases. The system automates Excel file analysis, schema inference, data validation, normalization recommendations, SQL script generation, and provides AI-powered migration guidance. The primary goal is to reduce manual effort while improving data quality and consistency.

## System Architecture
The application comprises six independent modules:

- **Schema Inference Module**: Automatically determines PostgreSQL column types, nullable constraints, and key candidates from uploaded datasets.
- **Validation Module**: Performs quality checks and identifies migration issues.
- **Data Analysis Module**: Recommends primary keys, unique constraints, and data quality improvements using a four-tier Candidate Key Quality model (

('# Summary: Data Migration Assistant – AI-Assisted Development Project\n\n## Project Overview\nData Migration Assistant is a web-based application designed to simplify migration of spreadsheet data into PostgreSQL databases. The system automates Excel file analysis, schema inference, data validation, normalization recommendations, SQL script generation, and provides AI-powered migration guidance. The primary goal is to reduce manual effort while improving data quality and consistency.\n\n## System Architecture\nThe application comprises six independent modules:\n\n- **Schema Inference Module**: Automatically determines PostgreSQL column types, nullable constraints, and key candidates from uploaded datasets.\n- **Validation Module**: Performs quality checks and identifies migration issues.\n- **Data Analysis Module**: Recommends primary keys, unique constraints, and data quality improvements using a four-tier Candidate Key Quality model (Strong, Plausible, Weak, None).\n- **Normalizati